# Comparing the performance of various methods on the forced Duffing oscillator

Forced Duffing (repo unforced Duffing + harmonic forcing): dx1 = x2; dx2 = -x1 - 0.1 x2 - 5 x1^3 + 5 cos(2t)

### Import the required libraries and files


In [ ]:
# Load ggplot2 first as it's a core dependency
library(ggplot2)
# Explicitly import ggproto
ggproto <- ggplot2::ggproto
Stat <- ggplot2::Stat

# Then load other visualization packages
library(plot3D)
library(tidyr)
library(tidyverse)
library(scales)
library(latex2exp)
library(cowplot)
library(ggh4x)
library(ggpubr)
library(gridExtra)
library(patchwork)
library(RColorBrewer)
library(stringr)
library(ggplotgui)
library(dplyr)

In [ ]:
nbfigsize <- function(width, height) {
    options(repr.plot.width = width, repr.plot.height = height, repr.plot.res = 300)
}

In [ ]:
setwd(dirname(dirname(getwd())))

In [ ]:
source("./MethodsEvaluation/results_processing_for_analysis.R")
source("./Pysindy/utils/pysindy_results_processing_for_analysis.R")

In [ ]:
ggplot_3d_path <- "./R/ggplot2-3d"
load_r <- list.files(ggplot_3d_path, "*.R")
sapply(load_r, function(i) {
    source(paste0(ggplot_3d_path, "/", i))
})

### General parameters settings


In [ ]:
dynamical_system_name <- "forced-duffing"
true_terms_1 <- list("x2")
true_terms_2 <- list("x1", "x2", "x1^3", "cos_2t")
true_terms_3 <- NULL
true_terms_4 <- NULL

### Success rate generation for various number of observations n


#### Load the argos family algorithms results


In [ ]:
root_path <- getwd()
experiment_name_list <- list("argos-alasso", "bayesian-alasso-ro")
method_list <- list("argos-alasso", "bayesian-alasso-ro")
function_number <- 2
exp_n <- TRUE
typical_pattern <- "snr49"
num_init <- 100
start <- 2
number_step <- 0.1

In [ ]:
total_success_rate_table_n <- generate_total_success_rate_table(
    root_path = root_path,
    experiment_name_list = experiment_name_list,
    method_list = method_list,
    function_number = function_number,
    num_init = num_init,
    dynamical_system_name = dynamical_system_name,
    exp_n = exp_n,
    typical_pattern = typical_pattern,
    true_terms_1 = true_terms_1,
    true_terms_2 = true_terms_2,
    true_terms_3 = true_terms_3,
    true_terms_4 = true_terms_4,
    start = start,
    number_step = number_step
)

tail(total_success_rate_table_n)

#### Load the pysindy algorithm results


In [ ]:
pysindy_path <- file.path(getwd(), "Pysindy")

In [ ]:
# Pysindy runs as SINDYc with control input u = cos(2t); the control feature
# is named "u0" and the PolynomialLibrary expands over [x0, x1, u0], so the
# library also contains u0 cross-terms (56 features total).
true_terms_1_py <- list("x1")
true_terms_2_py <- list("x0", "x1", "x0^3", "u0")
true_terms_3_py <- NULL
true_terms_4_py <- NULL

In [ ]:
dynamics_identification_results_n <- check_dynamical_system_py(
    root_path = pysindy_path,
    dynamical_system_name = dynamical_system_name,
    exp_n = exp_n,
    typical_pattern = typical_pattern,
    true_terms_1 = true_terms_1_py,
    true_terms_2 = true_terms_2_py,
    true_terms_3 = true_terms_3_py,
    true_terms_4 = true_terms_4_py,
    function_number = function_number
)

In [ ]:
pysindy_success_rate_table_n <- build_successful_rate_table_py(
    dynamics_identification_results = dynamics_identification_results_n,
    num_init = num_init,
    exp_n = exp_n,
    start = start,
    number_step = number_step,
    method = "pysindy"
)

In [ ]:
total_success_rate_table_n <- rbind(total_success_rate_table_n, pysindy_success_rate_table_n)
total_success_rate_table_n <- rename_model_names(total_success_rate_table_n)
tail(total_success_rate_table_n)

#### Plot the success rate for various n


##### General Plot settings for success rate


In [ ]:
ggplot_theme0 <- theme(
    axis.line = element_line(colour = "black"),
    axis.ticks.length = unit(.25, "cm"),
    panel.grid.major = element_blank(),
    panel.grid.minor = element_blank(),
    panel.border = element_blank(),
    panel.background = element_blank(),
    legend.key = element_blank(),
    legend.text = element_text(size = 20),
    # Changed for legend
    legend.title = element_text(face = "bold", size = 26),
    axis.text = element_text(size = 20),
    axis.title.x = element_text(size = 22),
    axis.title.y = element_text(
        size = 22,
        angle = 90,
        vjust = 0.5
    ),
    plot.title = element_text(size = 22),
    plot.tag = element_text(size = 28),
    legend.position = "none",
    legend.text.align = 0,
    plot.margin = margin(t = 0.75, r = 0.1, b = 0.25, l = 0.1, unit = "in")
)
ggplot_theme1 <- ggplot_theme0 + theme(plot.background = element_rect(fill = 0, colour = 0))

create_separators <- function(x, extra_x, y, extra_y, angle = 45, scale = 1, length = .1) {
    add_y <- length * sin(angle * pi / 180) / 2
    add_x <- length * cos(angle * pi / 180)
    list(
        x = x - add_x * scale, xend = x + add_x * scale + extra_x,
        y = rep(y - add_y * scale - extra_y, length(x)), yend = rep(y + add_y * scale - extra_y / 2, length(x))
    )
}

In [ ]:
# - Parameters settings for success rate plot (N)
total_correct <- total_success_rate_table_n
models_name <- unique(total_correct$Model)
new_levels <- models_name[c(1, 2, 3, 4, 5)]
total_correct$Model <- factor(total_correct$Model, levels = new_levels)
n_seq <- seq(2, 5, by = 0.1)
rect1 <- data.frame(xmin = -Inf, xmax = Inf, ymin = 0.8, ymax = Inf)
colors_correct <- c("#d74d3d", "#505ed5", "#8eb63d", "#FA5524", "#EF60AD")
shapes <- c(15, 16, 17, 18, 19)

x_labels <- c(
    expression(paste("10"^"2")),
    expression(paste("10"^"2.5")),
    expression(paste("10"^"3")),
    expression(paste("10"^"3.5")),
    expression(paste("10"^"4")),
    expression(paste("10"^"4.5")),
    expression(paste("10"^"5"))
)

x_breaks <- pretty_breaks()(n_seq)
y_breaks <-
    y_labels <- pretty_breaks()(c(0, max(total_correct$Value)))

prob_increase_n <-
    ggplot() +
    geom_hline(yintercept = 0.8, lty = 2) +
    geom_rect(data = rect1, aes(xmin = xmin, xmax = xmax, ymin = ymin, ymax = ymax), alpha = 0.3, fill = "#BDB6FB") +
    geom_point(
        data = total_correct,
        aes(
            x = eta,
            y = Value,
            fill = Model,
            col = Model,
            shape = Model
        ), size = 3
    ) +
    labs(
        y = "Success Rate",
        x = expression(italic("n"))
    ) +
    scale_x_continuous(
        labels = x_labels,
        breaks = x_breaks
    ) +
    scale_y_continuous(
        labels = y_labels,
        breaks = y_breaks,
        limits = c(0, max(y_labels))
    ) +
    ggplot_theme1 +
    # labs(title = paste(dynamical_system_name), tag = "a") +
    # theme(legend.position = "bottom") +
    scale_fill_manual(values = colors_correct, breaks = models_name, labels = models_name) +
    scale_colour_manual(values = colors_correct, breaks = models_name, labels = models_name) +
    scale_shape_manual(values = shapes, breaks = models_name, labels = models_name)

nbfigsize(
    width = 8,
    height = 6
)

# system_n <- arrangeGrob(prob_increase_n)
# grid.arrange(system_n, ncol = 1)
prob_increase_n

### Success rate generation for various snr values


In [ ]:
experiment_name_list <- list("argos-alasso", "bayesian-alasso-ro")
method_list <- list("argos-alasso", "bayesian-alasso-ro")
function_number <- 2
exp_n <- FALSE
typical_pattern <- "n5000"
num_init <- 100
start <- 1
number_step <- 1

# - Generate the total success rate table
total_success_rate_table_snr <- generate_total_success_rate_table(
    root_path = root_path,
    experiment_name_list = experiment_name_list,
    method_list = method_list,
    function_number = function_number,
    num_init = num_init,
    dynamical_system_name = dynamical_system_name,
    exp_n = exp_n,
    typical_pattern = typical_pattern,
    true_terms_1 = true_terms_1,
    true_terms_2 = true_terms_2,
    true_terms_3 = true_terms_3,
    true_terms_4 = NULL,
    start = start,
    number_step = number_step
)

tail(total_success_rate_table_snr)

In [ ]:
dynamics_identification_results_snr <- check_dynamical_system_py(
    root_path = pysindy_path,
    dynamical_system_name = dynamical_system_name,
    exp_n = exp_n,
    typical_pattern = typical_pattern,
    true_terms_1 = true_terms_1_py,
    true_terms_2 = true_terms_2_py,
    true_terms_3 = true_terms_3_py,
    true_terms_4 = true_terms_4_py,
    function_number = function_number
)

In [ ]:
pysindy_success_rate_table_snr <- build_successful_rate_table_py(
    dynamics_identification_results = dynamics_identification_results_snr,
    num_init = num_init,
    exp_n = exp_n,
    start = start,
    number_step = number_step,
    method = "pysindy"
)

head(pysindy_success_rate_table_snr)

In [ ]:
total_success_rate_table_snr <- rbind(total_success_rate_table_snr, pysindy_success_rate_table_snr)
total_success_rate_table_snr <- rename_model_names(total_success_rate_table_snr)
tail(total_success_rate_table_snr)

#### Plot the success rate based on various snr


In [ ]:
colors_correct <- c("#d74d3d", "#505ed5", "#8eb63d", "#FA5524", "#EF60AD")
shapes <- c(15, 16, 17, 18, 19)
rect1 <- data.frame(xmin = -Inf, xmax = Inf, ymin = 0.8, ymax = Inf)

total_correct <- total_success_rate_table_snr
for (i in seq(62, nrow(total_correct), 62)) {
    total_correct$snr[i] <- 73
}
models_name <- unique(total_correct$Model)
new_levels <- models_name[c(1, 2, 3, 4, 5)]
total_correct$Model <- factor(total_correct$Model, levels = new_levels)
# total_correct <- arrange(total_correct, Model)
# models_name <- new_levels[c(1, 2)]

x_labels <- x_breaks <- seq(1, 73, by = 12)
x_labels[length(x_labels)] <- TeX("$\\infty$")

y_labels <-
    y_breaks <-
    pretty_breaks()(c(min(total_correct$Value), max(total_correct$Value)))

xstart <- 65.5
xend <- 69.5
extra_x <- 1
y_sep <- min(total_correct$Value) - 0.05 * (min(total_correct$Value))
myseg <- create_separators(c(xstart, xend), extra_x = 1, y = y_sep, extra_y = 0.1, angle = 75)

prob_increase_snr <-
    ggplot() +
    geom_hline(yintercept = 0.8, lty = 2) +
    geom_rect(data = rect1, aes(xmin = xmin, xmax = xmax, ymin = ymin, ymax = ymax), alpha = 0.3, fill = "#BDB6FB") +
    geom_point(
        data = total_correct,
        aes(
            x = snr,
            y = Value,
            fill = Model,
            col = Model,
            shape = Model
        ), size = 3
    ) +
    labs(
        y = "Success Rate",
        x = TeX("SNR(dB)")
    ) +
    scale_x_continuous(
        limits = c(
            min(x_breaks),
            max(x_breaks)
        ),
        labels = x_labels,
        breaks = x_breaks
    ) +
    scale_y_continuous(
        labels = y_labels,
        breaks = y_breaks,
        limits = c(NA, max(y_breaks))
    ) +
    ggplot_theme1 +
    # labs(title = paste(dynamical_system_name), tag = "b") +
    theme(legend.position = "bottom") +
    scale_fill_manual(values = colors_correct, breaks = models_name, labels = models_name) +
    scale_colour_manual(values = colors_correct, breaks = models_name, labels = models_name) +
    scale_shape_manual(values = shapes, breaks = models_name, labels = models_name) +
    guides(x = guide_axis_truncated(
        trunc_lower = c(-Inf, xend + extra_x / 2),
        trunc_upper = c(xstart + extra_x / 2, Inf)
    )) +
    annotate("segment",
        x = myseg$x, xend = myseg$xend,
        y = myseg$y + 0.05, yend = myseg$yend
    ) +
    coord_cartesian(clip = "off", ylim = c(-0.0005, NA))

# prob_increase_snr + labs(title = paste(dynamical_system_name), tag = "b") + theme(legend.position = "bottom")
prob_increase_snr

In [ ]:
gglegend <- get_legend(prob_increase_snr)

# grid.arrange(system_n, ncol = 1)
system_comprehensive <-
    arrangeGrob(prob_increase_n + theme(legend.position = "none"),
        prob_increase_snr + theme(legend.position = "none"),
        # runtime_ggplot + theme(legend.position = "none"),
        nrow = 1
    )
nbfigsize(
    width = 16,
    height = 6
)
# system_comprehensive <- grid.arrange(system_comprehensive, gglegend, ncol = 1, heights = c(15, 1))
system_comprehensive <- grid.arrange(system_comprehensive, ncol = 1, heights = c(16, 0))
ggsave(plot = system_comprehensive, filename = sprintf(
        "./ResultsAnalysis/success-rate-plots/imgs/%s_success_rate_plot.pdf",
        dynamical_system_name
    ), width = 16, height = 6, dpi = 600, units = "in")
